# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides an example workflow for loading and exploring a FAIR^2 dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described using a Croissant schema. The schema URL is:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if necessary
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and explore the high-level summary using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show metadata description
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
List available record sets, their fields, and corresponding `@id` for reference.

The `@id` is used to uniquely reference each entity in the Croissant metadata.

In [ ]:
# List all record sets and their fields with @id's
print("Available record sets (@id and name):\n")
for record_set in dataset.metadata.recordSets:
    print(f"- {record_set.id} (name: {record_set.name})")
    if hasattr(record_set, 'fields'):
        for field in record_set.fields:
            col_list = [col.id for col in getattr(field, 'columns', [])] if hasattr(field, 'columns') else []
            print(f"    - field: {field.id} (name: {field.name}) columns: {col_list}")

# Save a list of record set @id's for easy reference later
record_set_ids = [rs.id for rs in dataset.metadata.recordSets]

# Show a quick preview of one record from the first record set
first_rs_id = record_set_ids[0] if record_set_ids else None
if first_rs_id:
    row = next(dataset.records(record_set=first_rs_id), None)
    print(f"\nSample record from {first_rs_id}:")
    pprint.pprint(row)

## 3. Data Extraction
Load tabular data from each record set into Pandas DataFrames, using record set and field `@id`s for reference.

In [ ]:
# Extract all record sets to DataFrames for easy exploration
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nRecord set: {rs_id}")
    print(f"Columns: {df.columns.to_list()}")
    print(df.head(2))

# Pick first record set for demo tasks below
main_rs = record_set_ids[0] if record_set_ids else None
df_main = dataframes[main_rs] if main_rs else pd.DataFrame()
print(f"\nUsing record set: {main_rs}")

## 4. Exploratory Data Analysis (EDA)
Demonstrate typical data processing by filtering a numeric field, normalizing its values, and grouping records by a categorical field.

**Note:** Use only field and column `@id`s for selection and processing.

In [ ]:
# Choose a numeric field and a group field using their @id (adjust if needed based on displayed fields above)
# For demonstration, guess likely candidates based on domain; adjust according to actual IDs in your data
print("\nAvailable columns in the main record set:")
print(df_main.columns.to_list())

# Example guess: 'ucr:age_at_second_crc' as numeric field, 'ucr:sex' as group field (replace with actual IDs you see above)
numeric_field = None
group_field = None
# Try to let the user identify which columns can be numeric/group
for col in df_main.columns:
    if 'age' in col or 'interval' in col or 'years' in col:
        numeric_field = col
    if 'sex' in col or 'gender' in col:
        group_field = col

if not numeric_field:
    numeric_field = df_main.select_dtypes(include='number').columns[0] if not df_main.empty else None

if not group_field:
    # Fallback to first string field
    for col in df_main.columns:
        if df_main[col].dtype == 'object':
            group_field = col
            break

print(f"\nNumeric field chosen: {numeric_field}")
print(f"Group field chosen: {group_field}")

# Demo: Filter, normalize, and group
if numeric_field and (numeric_field in df_main.columns):
    threshold = df_main[numeric_field].mean() if pd.api.types.is_numeric_dtype(df_main[numeric_field]) else 0
    filtered_df = df_main[df_main[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold:.1f}:")
    print(filtered_df.head())
    # Normalize (z-score)
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} (first 5 records):")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Group
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        print(grouped_df)
else:
    print("Could not identify a suitable numeric field for processing.")

## 5. Visualization
Plot distributions and relationships between fields, referencing with their `@id`s where possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in df_main.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df_main[numeric_field], bins=8, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field and group_field in df_main.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=df_main[group_field], y=df_main[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, process, and visualize data from a FAIR^2-compliant tabular dataset using the `mlcroissant` library. All references to dataset components rely on their unique `@id` as defined by the Croissant schema. You can adapt this workflow for further statistical analysis or machine learning tasks.